# 4 Results and analysis

Reads every finished run under `hefl/results/` and builds the paper tables. Nothing here trains anything, so it is instant and safe to re-run.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np, torch, matplotlib.pyplot as plt
from hefl.utils import pick_device, set_seed
set_seed(42); DEVICE = pick_device('auto')
print('device:', DEVICE)


In [ ]:
!cd .. && python -m hefl.aggregate


## Per-rotation accuracy

The **diagonal** of the per-expert block is the load-bearing number: does expert *k* win on rotation *k*? That validates clustering and specialisation in one measurement.


In [ ]:
import json, glob
runs = {}
for f in sorted(glob.glob('../hefl/results/*/results.json')):
    runs[f.split('/')[-2]] = json.load(open(f))
print('finished runs:', list(runs) or '(none yet - run notebook 03 or the sweep)')


In [ ]:
name = next(iter(runs))
res = runs[name]
mat = np.array(res['expert_matrix'])
groups = res['config']['rotation_groups']

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(mat, cmap='viridis')
ax.set_xticks(range(len(groups)), [f'{g*90}°' for g in groups])
ax.set_yticks(range(mat.shape[0]), [f'expert {k}' for k in range(mat.shape[0])])
ax.set_xlabel('test rotation'); ax.set_title(f'Per-expert accuracy — {name}')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f'{mat[i,j]:.3f}', ha='center', va='center',
                color='white' if mat[i,j] < mat.max()*0.7 else 'black', fontsize=9)
plt.colorbar(im, label='accuracy'); plt.tight_layout(); plt.show()
print('Specialisation shows up as a bright diagonal after permuting columns by cluster→rotation.')


## Method comparison


In [ ]:
methods, scores = [], []
for key, label in (('centralized','Centralized (ceiling)'), ('fedavg','FedAvg'),
                   ('ensemble_uniform','Ensemble (uniform)'), ('ensemble_beta','Ensemble (β)'),
                   ('ensemble_gate','Ensemble (gate)'), ('ensemble_mlp','Ensemble (MLP)'),
                   ('oracle','Oracle expert')):
    v = res.get(key, {}).get('test', {}).get('overall')
    if v is not None: methods.append(label); scores.append(v)

fig, ax = plt.subplots(figsize=(8, 3.6))
colors = ['#888' if 'Ceiling' in m or 'Oracle' in m else '#2a6fb0' for m in methods]
ax.barh(methods, scores, color=colors)
ax.set_xlabel('overall accuracy'); ax.set_title(f'Method comparison — {name}')
for i, v in enumerate(scores): ax.text(v, i, f'  {v:.4f}', va='center', fontsize=9)
ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show()


## Training curves


In [ ]:
hist = [h for h in res['history'] if h['kind'] == 'expert']
rounds = [h['round'] for h in hist]; loss = [h['mean_local_loss'] for h in hist]
ev = [(h['round'], h['val_accuracy']) for h in hist if h.get('val_accuracy') is not None]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.plot(rounds, loss, lw=1.5); a1.axhline(np.log(10), color='r', ls=':', label='chance (ln 10)')
a1.set_xlabel('round'); a1.set_ylabel('mean local loss'); a1.legend(); a1.grid(alpha=0.3)
if ev:
    a2.plot([e[0] for e in ev], [e[1] for e in ev], 'o-')
    a2.axhline(0.1, color='r', ls=':', label='chance')
    a2.set_xlabel('round'); a2.set_ylabel('validation accuracy'); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Deployment question: how many experts do you actually need?

Requires a run with `subset_analysis: true`. Every subset is magnitude-correct by construction, so these numbers are directly comparable.


In [ ]:
subs = res.get('subsets')
if not subs:
    print('no subset analysis in this run (set subset_analysis: true)')
else:
    best = {}
    for r in subs: best[r['size']] = max(best.get(r['size'], 0), r['overall'])
    sizes = sorted(best)
    plt.figure(figsize=(6, 3.6))
    plt.plot(sizes, [best[s] for s in sizes], 'o-', lw=2)
    plt.xlabel('number of active experts'); plt.ylabel('best overall accuracy')
    plt.title('Accuracy vs inference cost'); plt.grid(alpha=0.3); plt.xticks(sizes)
    plt.tight_layout(); plt.show()
